# Deepchecks Demo

In [ ]:
tensors
stringhe => path a imgs e audio
spettrogrammi


Immagini:
    - luminosità =>

## QA for tabular data

In [ ]:
from deepchecks.tabular.datasets.classification import iris
from deepchecks.tabular import Dataset
from sklearn.model_selection import train_test_split

# Load Data
iris_df = iris.load_data(data_format="Dataframe", as_train_test=False)
iris_df.head()

In [2]:
label_col = 'target'
df_train, df_test = train_test_split(iris_df, stratify=iris_df[label_col], random_state=0)

In [3]:
ds_train = Dataset(df_train, label=label_col, cat_features=[])
ds_test =  Dataset(df_test,  label=label_col, cat_features=[])

**N.B.**: We explicitly state that this dataset has no categorical features, otherwise they will be automatically inferred.

If the dataset has categorical features, the best practice is to pass a list with their names

### Running individual Deepchecks checks

First, we can try running individual checks offered by Deepchecks.

For tabular data, we have the following groups of checks:

- [data integrity checks](https://docs.deepchecks.com/stable/tabular/auto_checks/data_integrity/index.html#)
- [train-test validation checks](https://docs.deepchecks.com/stable/tabular/auto_checks/train_test_validation/index.html)
- [model evaluation checks](https://docs.deepchecks.com/stable/tabular/auto_checks/model_evaluation/index.html)

Let's explore a couple of checks from the data integity group: `ColumnsInfo` and `DataDuplicates`.

#### Columns Info

In [ ]:
from deepchecks.tabular.checks.data_integrity import ColumnsInfo

check = ColumnsInfo()
check.run(ds_train)

#### Data Duplicates

In [ ]:
from deepchecks.tabular.checks.data_integrity import DataDuplicates

DataDuplicates().run(ds_train)

### Running a Deepchecks suite

Beyond individual checks, we can try running one of the several suites of checks offered by Deepchecks.

Here, again, we focus on data integrity, thus we run the `data_integrity` suite.

In [ ]:
from deepchecks.tabular.suites import data_integrity

integ_suite = data_integrity()
integ_suite.run(ds_train)

### Customizing a Deepchecks suite

Existing Deepchecks suites can be easily customized. E.g., checks can be added or removed; check conditions can be added, edited, or removed as well.

Let's make an example by removing a check from the suite we just ran.

In [ ]:
# Lets first print the suite to find the conditions that we want to change:

integ_suite

In [ ]:
print(integ_suite[9])

In [10]:
integ_suite[9].remove_condition(0)

In [ ]:
print(integ_suite[9])

In [ ]:
integ_suite.run(ds_train)

### Composing a custom suite

Alternatively to customizing an existing suite, we can compose a custom suite from scratch.

By using the `Suite` constructor from the right Deepchecks module (i.e., `tabular`, `nlp`, or `vision`), we just need to provide a name for the custom suite and list the checks we want to include.

In [13]:
from deepchecks.tabular import Suite
from deepchecks.tabular.checks.data_integrity import (
    FeatureLabelCorrelation,
    FeatureFeatureCorrelation,
)

custom_suite = Suite(
    "Correlation checks",
    FeatureLabelCorrelation().add_condition_feature_pps_less_than(0.7),
    FeatureFeatureCorrelation().add_condition_max_number_of_pairs_above_threshold(0.8),
)

In [ ]:
custom_suite

In [ ]:
custom_suite.run(ds_train)

## QA for image data

Differently from Great Expectations, Deepchecks is not limited to textual data.
It can also be used tu assure the quality of textual data and image data.

Let's make an example with a dataset of images.

To use Deepchecks for image data QA, we need to install a few additional dependencies:
- `pip install "deepchecks[vision]"`
- `pip install torchvision`

N.B.: `torchvision` version is required to be >=0.11.3

For this tutorial, we’ll be using a small sample of the [RGB EuroSAT dataset](https://github.com/phelber/eurosat#). EuroSAT dataset is based on Sentinel-2 satellite images covering 13 spectral bands and consisting of 10 classes with 27000 labeled and geo-referenced samples.

In [15]:
from deepchecks.vision.checks import ImageDatasetDrift
from deepchecks.vision.suites import train_test_validation
from deepchecks.vision import classification_dataset_from_directory

In [18]:
train_ds, test_ds = classification_dataset_from_directory(
    root='C:\\Users\\ivanr\\Desktop\\art2cook\\data\\images\\', object_type='VisionData', image_extension='jpg')

In [19]:
check = ImageDatasetDrift()
result = check.run(train_dataset=train_ds, test_dataset=test_ds)
result

Now we can run all the available [Deepchecks' checks and suites for image data](https://docs.deepchecks.com/stable/vision/index.html#vision-checks-gallery).
Let's try with a  train_test_validation suite:

In [20]:
suite = train_test_validation()
result = suite.run(train_ds, test_ds)

In [21]:
result.show()

Accordion(children=(VBox(children=(HTML(value='\n<h1 id="summary_XAJI0Y6DPBHSAHXTHV3A3ZMF8">Train Test Validat…